# EpiScope Tutorial (Reconstructed): Precision Miner

This reconstructed notebook replaces the original, which could not be parsed due to invalid JSON (embedded control characters and truncated strings).

It shows a **self-contained precision mining** workflow:

- Ingest a small set of mock abstracts
- Extract simple entities (dates, drugs, viruses) with regex
- Score relevance to a user query
- Produce a compact, structured result object

> No external services (GROBID, Qdrant, LLMs) are required.

In [1]:
import re
from dataclasses import dataclass
from typing import List, Dict, Any

DOCS = [
    {
        "id": "d1",
        "title": "Effectiveness of Flu Vaccination",
        "abstract": "A 2023 study shows annual influenza vaccination reduces hospitalizations in older adults.",
    },
    {
        "id": "d2",
        "title": "Oseltamivir in Early Treatment",
        "abstract": "Early oseltamivir reduced illness duration in high-risk patients within 48 hours of symptom onset.",
    },
    {
        "id": "d3",
        "title": "COVID-19 Non-Pharmaceutical Interventions",
        "abstract": "Masking and hand hygiene were associated with lower transmission rates in community studies.",
    },
]

DRUGS = {"oseltamivir", "remdesivir", "nirmatrelvir", "ritonavir"}
VIRUSES = {"influenza", "covid-19", "sars-cov-2"}
DATE_RE = re.compile(r"\b(19|20)\d{2}\b")

@dataclass
class ExtractionResult:
    doc_id: str
    title: str
    date_years: List[str]
    drugs: List[str]
    viruses: List[str]
    relevance: float

def extract(doc: Dict[str, Any], query: str):
    text = f"{doc['title']} {doc['abstract']}".lower()
    years = DATE_RE.findall(text)
    drugs = sorted([d for d in DRUGS if d in text])
    viruses = sorted([v for v in VIRUSES if v in text])
    # trivial relevance: count of query tokens present
    q_tokens = {w for w in re.findall(r"[a-z0-9-]+", query.lower()) if len(w) > 2}
    score = sum(text.count(w) for w in q_tokens)
    return ExtractionResult(doc_id=doc["id"], title=doc["title"], date_years=years, drugs=drugs, viruses=viruses, relevance=float(score))

def run_pipeline(query: str) -> List[ExtractionResult]:
    return sorted([extract(d, query) for d in DOCS], key=lambda r: r.relevance, reverse=True)

print("Precision miner ready.")

Precision miner ready.


In [2]:
QUERY = "influenza hospitalization oseltamivir"
results = run_pipeline(QUERY)
for r in results:
    print(f"Doc {r.doc_id}: {r.title}")
    print("  Years:", r.date_years)
    print("  Drugs:", r.drugs)
    print("  Viruses:", r.viruses)
    print("  Relevance:", r.relevance)
    print()

Doc d1: Effectiveness of Flu Vaccination
  Years: ['20']
  Drugs: []
  Viruses: ['influenza']
  Relevance: 2.0

Doc d2: Oseltamivir in Early Treatment
  Years: []
  Drugs: ['oseltamivir']
  Viruses: []
  Relevance: 2.0

Doc d3: COVID-19 Non-Pharmaceutical Interventions
  Years: []
  Drugs: []
  Viruses: ['covid-19']
  Relevance: 0.0

